## Imports

In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler, LabelEncoder

## Loading the dataset

In [2]:
df = pd.read_csv('../data/raw/insurance.csv')
df.head()

,id,Gender,Age,Driving_License,Region_Code,Previously_Insured,Vehicle_Age,Vehicle_Damage,Annual_Premium,Policy_Sales_Channel,Vintage,Response
0,1,Male,44,1,28.0,0,> 2 Years,Yes,40454.0,26.0,217,1
1,2,Male,76,1,3.0,0,1-2 Year,No,33536.0,26.0,183,0
2,3,Male,47,1,28.0,0,> 2 Years,Yes,38294.0,26.0,27,1
3,4,Male,21,1,11.0,1,< 1 Year,No,28619.0,152.0,203,0
4,5,Female,29,1,41.0,1,< 1 Year,No,27496.0,152.0,39,0


In [3]:
# Check number of rows
rows = df.shape[0]
rows

381109

In [4]:
# Check for missing values
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 381109 entries, 0 to 381108
Data columns (total 12 columns):
 #   Column                Non-Null Count   Dtype  
---  ------                --------------   -----  
 0   id                    381109 non-null  int64  
 1   Gender                381109 non-null  object 
 2   Age                   381109 non-null  int64  
 3   Driving_License       381109 non-null  int64  
 4   Region_Code           381109 non-null  float64
 5   Previously_Insured    381109 non-null  int64  
 6   Vehicle_Age           381109 non-null  object 
 7   Vehicle_Damage        381109 non-null  object 
 8   Annual_Premium        381109 non-null  float64
 9   Policy_Sales_Channel  381109 non-null  float64
 10  Vintage               381109 non-null  int64  
 11  Response              381109 non-null  int64  
dtypes: float64(3), int64(6), object(3)
memory usage: 34.9+ MB


In [5]:
# Check for implicit missing values
print(np.unique(df["Gender"]))
print(np.unique(df["Vehicle_Age"]))
print(np.unique(df["Vehicle_Damage"]))

['Female' 'Male']
['1-2 Year' '< 1 Year' '> 2 Years']
['No' 'Yes']


It looks like there are no explicit or implicit missing values, meaning we won't have to drop any rows of the dataset. However, we will drop id, as that is not an important feature for us.

In [6]:
df = df.drop('id', axis=1)
df.head()

,Gender,Age,Driving_License,Region_Code,Previously_Insured,Vehicle_Age,Vehicle_Damage,Annual_Premium,Policy_Sales_Channel,Vintage,Response
0,Male,44,1,28.0,0,> 2 Years,Yes,40454.0,26.0,217,1
1,Male,76,1,3.0,0,1-2 Year,No,33536.0,26.0,183,0
2,Male,47,1,28.0,0,> 2 Years,Yes,38294.0,26.0,27,1
3,Male,21,1,11.0,1,< 1 Year,No,28619.0,152.0,203,0
4,Female,29,1,41.0,1,< 1 Year,No,27496.0,152.0,39,0


## Preprocessing

In [7]:
# Handle categorical data
df_processed = df
one_hot_cols = ['Gender', 'Vehicle_Damage']
df_processed = pd.get_dummies(df_processed, columns=one_hot_cols, drop_first=True)

In [8]:
# Ordinal encode 'Vehicle_Age'
vehicle_age_mapping = {'< 1 Year': 0, '1-2 Year': 1, '> 2 Years': 2}
df_processed['Vehicle_Age'] = df_processed['Vehicle_Age'].map(vehicle_age_mapping)
df_processed.head()

,Age,Driving_License,Region_Code,Previously_Insured,Vehicle_Age,Annual_Premium,Policy_Sales_Channel,Vintage,Response,Gender_Male,Vehicle_Damage_Yes
0,44,1,28.0,0,2,40454.0,26.0,217,1,True,True
1,76,1,3.0,0,1,33536.0,26.0,183,0,True,False
2,47,1,28.0,0,2,38294.0,26.0,27,1,True,True
3,21,1,11.0,1,0,28619.0,152.0,203,0,True,False
4,29,1,41.0,1,0,27496.0,152.0,39,0,False,False


In [9]:
df_processed.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 381109 entries, 0 to 381108
Data columns (total 11 columns):
 #   Column                Non-Null Count   Dtype  
---  ------                --------------   -----  
 0   Age                   381109 non-null  int64  
 1   Driving_License       381109 non-null  int64  
 2   Region_Code           381109 non-null  float64
 3   Previously_Insured    381109 non-null  int64  
 4   Vehicle_Age           381109 non-null  int64  
 5   Annual_Premium        381109 non-null  float64
 6   Policy_Sales_Channel  381109 non-null  float64
 7   Vintage               381109 non-null  int64  
 8   Response              381109 non-null  int64  
 9   Gender_Male           381109 non-null  bool   
 10  Vehicle_Damage_Yes    381109 non-null  bool   
dtypes: bool(2), float64(3), int64(6)
memory usage: 26.9 MB


In [10]:
# Standardize numerical features
scaler = StandardScaler()
numerical_cols = ['Age', 'Annual_Premium', 'Vintage']
df_processed[numerical_cols] = scaler.fit_transform(df_processed[numerical_cols])

We still have Region_Code and Policy_Sales_Channel, which are represented as numeric data types, but are by definition categories. We will handle this by using embedding layers, which is useful when we have high cardinality categorical features. These layers will be defined layer, but I first need to process the data her by making sure they data is encoded as integers.

In [11]:
# Initialize label encoders
le_region = LabelEncoder()
le_policy = LabelEncoder()

# Fit and transform the categorical features
df_processed['Region_Code'] = le_region.fit_transform(df_processed['Region_Code'])
df_processed['Policy_Sales_Channel'] = le_policy.fit_transform(df_processed['Policy_Sales_Channel'])

Now we have handled our categorical features and have scaled our numerical data

## Saving the data

In [13]:
# Save processed data
df.to_csv('../data/processed/insurance_clean.csv', index=False)
df_processed.to_csv('../data/processed/insurance_processed.csv', index=False)